# Projet 5MA-OSI : Transfert d'organes en domino sous incertitudes

## Introduction

Malgré l'augmentation croissante du nombre de transplantations d'organes effectuées chaque année (environ 6000 en 2017 dont 3782 transplantations de reins), la demande reste en perpétuelle augmentation. Ainsi 6000 organes, dont 3782 reins, ont été transplantés en 2017, mais il y avait encore 24000 personnes en attente d'un organe la même année. Les organes transplantés peuvent provenir d'un donneur décédé ou, dans le cas des reins et du foie, d'un donneur vivant consentant, le plus souvent membre de la famille du patient. Hélas, même si un proche accepte de prendre ce risque pour sa santé, il ne sera pas forcément compatible avec le patient. Pour cette raison, les pratiques médicales et les législations évoluent dans de nombreux pays afin de permettre la mise en place d'un programme d'échange de dons d'organes.

L'exemple le plus simple d'échange de don d'organes est celui où deux patients $P_1$ et $P_2$ sont accompagnés de donneurs $D_1$ et $D_2$. Les patients sont supposés incompatibles avec les donneurs qui les accompagnent, mais on suppose que $D_1$ est compatible avec $P_2$ et $D_2$ avec $P_1$. Il est alors possible de transplanter un organe de $D_1$ vers $P_2$ et de $D_2$ vers $P_1$ avec le consentement de tous et en suivant la procédure légale.

Plus généralement, un cycle d'échange d'organes associe $k$ paires de patient-donneur $(P_{i_1},D_{i_1}), \dots, (P_{i_k},D_{i_k})$ de sorte que $D_{i_l}$ donne à $P_{i_{l+1}}$ pour $l=1,\dots,k-1$ et $D_{i_k}$ donne à $P_{i_1}$.
Par ailleurs, le point essentiel est que les transferts soient tous réalisés en même temps et dans le même hôpital pour éviter qu'une rétractation de dernière minute ne lèse un patient et son donneur, et que les patients et donneurs venus ensemble et leur famille puissent se soutenir émotionnellement durant l'hospitalisation. 
Pour cette raison, le nombre d'échanges prenant place au sein d'un même cycle est nécessairement limité. En pratique, l'organisation d'un cycle de trois paires est déjà une épreuve pour le personnel d'un hôpital, et le plus grand cycle ayant jamais eu lieu a a impliqué six patients et donneurs.

Dans ce projet, nous prendrons le point de vue de l'organisme national responsable de la gestion du programme d'échange d'organes. 
À chaque phase d'échange, l'objectif de cet organisme est de choisir un ensemble de cycles d'échanges entre paires compatibles afin de maximiser le nombre de patients recevant un organe. Dans certains cas, on peut aussi donner une priorité à certains patients en fonction de la gravité de leur état ou de la durée de leur attente. 
Pour cela, on pourra attribuer des poids différents à chaque patient et maximiser la somme des poids des patients recevant un organe. 

## Étude du problème déterministe


### Jeux de données

Tous vos tests seront basés sur les jeux de données de la [PrefLib](https://www.preflib.org/dataset/00036). Ces jeux de données ne correspondent pas à des programmes d'échanges d'organes réels pour des raisons de confidentialité, mais ils reproduisent la structure des données réelles plus fidèlement que les données aléatoires utilisées pour le projet de RO. Ils 
sont constituées d'informations individuelles et d'un graphe de compatibilité. Chaque fichier .wmd décrit un graphe de compatibilité orienté, $G=(V,A)$, où chaque sommet de $V$ représente une paire patient-donneur et où un arc entre deux paires $(P_k,D_k)$ et $(P_l,D_l)$ signifie que $D_k$ est compatible avec $P_l$. La compatibilité est obtenue à partir des données biologiques individuelles (e.g., les groupes sanguins) et d'un _test croisé_ lors duquel des biologistes mettent en présence des tissus d'un malade et d'un donneur supposé. Chaque fichier est composé comme suit :  
- les 11 premières ligne contiennent diverses informations sur le fichier, dont le nombre de sommets ($n$) et le nombre d'arcs ($m$),
- les $n$ lignes suivantes nomment les sommets (un sommet par paire),
- les $m$ lignes restantes contiennent les arcs du graphe et le poids de chaque arc. Le poids de chaque arc indique l'urgence de la situation du malade de la paire destination. 

Je fournis ci-dessous une fonction permettant de lire les fichiers .wmd et retournant un graphe de compatibilité pondéré par les poids de chaque patient. Pour vous aider à vous lancer, je fournis déjà quelques instances en accompagnement de ce sujet sur Moodle. Elles contiennent un nombre croissant de paires patient-donneur allant de 16 à 512.

In [1]:
# Include the necessary packages
using Random, Graphs, JuMP, HiGHS, DelimitedFiles, Distributions
include("KEP_readfile.jl")

read_wmd_file

In [2]:
# Exemple d'utilisation de la fonction supposant que les fichiers sont dans le dossier data_KEP
G, edge_weight = read_wmd_file("data_KEP/KEP_071.wmd");
println("Number of vertices: ", nv(G));
println("Number of arcs: ", ne(G));
println("List of arcs with weights:")
for e in edges(G)
    print("($(e.src),$(e.dst)):$(edge_weight[e.src,e.dst]), ");
end

Number of vertices: 64
Number of arcs: 1191
List of arcs with weights:
(1,14):1.0, (1,16):1.0, (1,17):1.0, (1,18):1.0, (1,37):1.0, (1,40):1.0, (1,51):1.0, (1,54):1.0, (1,57):1.0, (1,60):1.0, (2,1):1.0, (2,3):1.0, (2,4):1.0, (2,5):1.0, (2,6):1.0, (2,7):1.0, (2,9):1.0, (2,10):1.0, (2,11):1.0, (2,12):1.0, (2,14):1.0, (2,15):1.0, (2,16):1.0, (2,17):1.0, (2,20):1.0, (2,21):1.0, (2,22):1.0, (2,23):1.0, (2,24):1.0, (2,29):1.0, (2,35):1.0, (2,36):1.0, (2,37):1.0, (2,40):1.0, (2,42):1.0, (2,46):1.0, (2,48):1.0, (2,49):1.0, (2,50):1.0, (2,53):1.0, (2,55):1.0, (2,57):1.0, (2,58):1.0, (2,59):1.0, (2,60):1.0, (2,61):1.0, (2,62):1.0, (2,64):1.0, (3,14):1.0, (3,16):1.0, (3,17):1.0, (3,18):1.0, (3,36):1.0, (3,37):1.0, (3,38):1.0, (3,40):1.0, (3,46):1.0, (3,57):1.0, (3,60):1.0, (3,64):1.0, (4,11):1.0, (4,14):1.0, (4,16):1.0, (4,17):1.0, (4,37):1.0, (4,46):1.0, (4,52):1.0, (4,57):1.0, (4,60):1.0, (4,64):1.0, (5,11):1.0, (5,16):1.0, (5,17):1.0, (5,38):1.0, (5,40):1.0, (5,46):1.0, (5,54):1.0, (5,57):1.0, 

### Formulations compactes

Dans le cadre du TD/TP du cours de Recherche Opérationnelle, il vous a été demandé de coder trois formulations PLNE, à savoir :
- celle sans contrainte sur la longueur des cycles
- celle avec des cycles de taille 2
- celle avec la décomposition par hôpital pour des cycles de taille 2 à 6.

J'ai recopié la correction distribuée sur Moodle dans le fichier formulation_compactes.jl. Je l'inclus ci dessous. Afin de prendre le sujet en main, comparer les solutions des 3 modèles sur un ensemble de jeux de données bien choisis. Analyser les résultats d'un point de vue informatique, pratique et sociétal.

In [4]:
# inclusion des fonctions de résolution des formulations compactes
include("formulations_compactes.jl")

# tests de bon fonctionnement sur le petit graphe importé plus haut
# solve_cycle_infini(G, edge_weight);
# solve_cycle_2(G, edge_weight);
G, edge_weight = read_wmd_file("data_KEP/KEP_071.wmd");
K = 3
solve_cycle_K(G, edge_weight, K);


-------------------------------
Résolution du modèle avec cycle de taille au plus 3

Statut de la résolution : OPTIMAL
Durée de la résolution : 5.3977628
Nombre de transferts réalisés : 47
Liste des transferts réalisés :
2 -> 23 ; 4 -> 17 ; 5 -> 60 ; 6 -> 40 ; 8 -> 41 ; 10 -> 36 ; 11 -> 62 ; 12 -> 57 ; 13 -> 48 ; 14 -> 15 ; 15 -> 27 ; 17 -> 4 ; 18 -> 43 ; 19 -> 33 ; 20 -> 51 ; 22 -> 45 ; 23 -> 2 ; 24 -> 13 ; 25 -> 28 ; 27 -> 14 ; 28 -> 25 ; 30 -> 56 ; 32 -> 35 ; 33 -> 46 ; 35 -> 32 ; 36 -> 63 ; 37 -> 49 ; 38 -> 37 ; 40 -> 61 ; 41 -> 8 ; 42 -> 64 ; 43 -> 52 ; 45 -> 22 ; 46 -> 19 ; 48 -> 24 ; 49 -> 38 ; 51 -> 20 ; 52 -> 18 ; 54 -> 55 ; 55 -> 54 ; 56 -> 30 ; 57 -> 12 ; 60 -> 5 ; 61 -> 6 ; 62 -> 11 ; 63 -> 10 ; 64 -> 42 ; 
-------------------------------



### Génération de colonnes

L'objectif de cette partie du projet est de coder une méthode de génération de colonnes pour le problème de dons d'organes en dominos (KEP pour Kidney Exchange Problem en anglais) sans incertitudes. Votre code s'appuiera sur les éléments vus en CM et leur application au KEP vue en TD. Pour compléter cette présentation, [une page de la documentation de JuMP](https://jump.dev/JuMP.jl/stable/tutorials/algorithms/cutting_stock_column_generation/) est dédiée à la génération de colonnes. Elle vous offre une autre entrée en matière sur la question.

1. Coder une fonction résolvant le modèle par cycles après énumération de touts les cycles de taille $K$ ou moins. Tester pour $K=2,3,4$.
2. Coder une méthode de génération de colonnes pour résoudre la relaxation linéaire de la formulation par cycles. Vous implémenterez deux versions de la fonction de _pricing_ (résolution du sous-problème) : une ou le sous-problème est formulé à l'aide d'un ensemble de PLNE, et une ou le sous-problème est résolu par un algorithme de plus long chemin.
3. Coder une fonction qui résout le problème de dons en dominos de façon approchée à l'aide de l'heuristique de génération de colonnes décrite en CM.


## 1. brute force approach

In [3]:
include("brute_force_approach.jl")
K = 5
G, edge_weight = read_wmd_file("data_KEP/KEP_031.wmd");
C_K = enumerate_cycles(G,K)

brut_force(G,K)

LoadError: LoadError: UndefVarError: `@variable` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/jeanpousset/Documents/7-Optimisation Stochastique/Projet - Transfert d'organe/brute_force_approach.jl:52
in expression starting at /Users/jeanpousset/Documents/7-Optimisation Stochastique/Projet - Transfert d'organe/brute_force_approach.jl:52

## 2,3. Column generation for Kidney Exchange Problems

In [6]:
using Pkg
Pkg.add("CSV")
using CSV

   Resolving package versions...
    Updating `C:\Users\Yilon\.julia\environments\v1.12\Project.toml`
  [336ed68f] + CSV v0.10.15
    Manifest No packages added to or removed from `C:\Users\Yilon\.julia\environments\v1.12\Manifest.toml`


In [8]:
using Pkg
Pkg.add("DataFrames")

   Resolving package versions...
   Installed InvertedIndices ──── v1.3.1
   Installed Crayons ──────────── v4.1.1
   Installed StringManipulation ─ v0.4.2
   Installed SnoopPrecompile ──── v1.0.3
   Installed PrettyTables ─────── v2.4.0
   Installed DataFrames ───────── v1.5.0
    Updating `C:\Users\Yilon\.julia\environments\v1.12\Project.toml`
⌃ [a93c6f00] + DataFrames v1.5.0
    Updating `C:\Users\Yilon\.julia\environments\v1.12\Manifest.toml`
  [a8cc5b0e] + Crayons v4.1.1
⌃ [a93c6f00] + DataFrames v1.5.0
  [41ab1584] + InvertedIndices v1.3.1
⌅ [08abe8d2] + PrettyTables v2.4.0
  [66db9d55] + SnoopPrecompile v1.0.3
  [892a3eda] + StringManipulation v0.4.2
        Info Packages marked with ⌃ and ⌅ have new versions available. Those with ⌃ may be upgradable, but those with ⌅ are restricted by compatibility constraints from upgrading. To see why use `status --outdated -m`
Precompiling packages...
    640.5 ms  ✓ InvertedIndices
    593.6 ms  ✓ SnoopPrecompile
   1843.3 ms  ✓ StringManip

In [120]:
using Random, Graphs, JuMP, HiGHS, DelimitedFiles, Distributions

include("tests.jl")

test01 = KEP_test("data_KEP/KEP_191.wmd"; SP_method="Bellmann")

transferts = solve_KEP(test01)[1]


┌ Warning: To use these tests, you need to download necessary the KEP data available on Preflib (Kidney Data 00036) 
└ @ Main C:\Users\Yilon\Desktop\Optimisation sous Incertitude\Stochastic-programming-KEP-Julia-main\Stochastic-programming-KEP-Julia-main\tests.jl:8


159-element Vector{Vector{Int64}}:
 [120, 449]
 [386, 398]
 [422, 508]
 [327, 403]
 [14, 203]
 [320, 438]
 [200, 502]
 [33, 228]
 [221, 326]
 [2, 231]
 [168, 459]
 [60, 276]
 [41, 239]
 ⋮
 [139, 253, 169, 164]
 [72, 364, 95, 73]
 [238, 488, 249, 315]
 [366, 467, 377, 471]
 [224, 357, 259, 260]
 [106, 187]
 [384, 415, 418, 432]
 [10, 308, 460, 328]
 [152, 257, 165, 245]
 [433, 446, 442, 472]
 [92, 324, 289, 192]
 [339, 391, 409, 461]

In [12]:
transferts

159-element Vector{Vector{Int64}}:
 [149, 456]
 [77, 242]
 [162, 369]
 [449, 510]
 [160, 328]
 [204, 231]
 [421, 508]
 [377, 434]
 [279, 349]
 [220, 502]
 [51, 161]
 [198, 319]
 [216, 240]
 ⋮
 [208, 223]
 [3, 89]
 [333, 364, 438, 338]
 [414, 425, 452]
 [105, 197, 125, 176]
 [248, 251, 433, 270]
 [92, 175, 139, 192]
 [256, 357, 259, 308]
 [112, 165, 245]
 [14, 254, 249, 19]
 [84, 215, 384, 95]
 [260, 261, 307]

In [5]:
"""
    gs_compatible

Renvoie true si un groupe sanguin de donneur est compatible avec celui d'un malade 

# Paramètres
* `gsd::String` : groupe sanguin d'un donneur
* `gsm::String` : groupe sanguin d'un malade
"""
function gs_compatible(gsd::String, gsm::String)
    if (gsd == "O") || (gsm == "AB")
        # O peut donner à {O,A,B,AB}, AB peut recevoir de {O,A,B,AB}
        return true
    elseif (gsd == "A") && (gsm == "A")
        #  A donne à {A,AB}
        return true
    elseif (gsd == "B") && (gsm == "B")
        # B donne à {B,AB}
        return true
    else
        # O ne peut pas recevoir de {A,B,AB}, A ne peut pas recevoir {B,AB}, B ne peut pas recevoir de {A,AB}
        return false;
    end
end

"""
    read_dat_file

Contruct Gtilde from a `.dat` file from PrefLib.

# Parameters
* `dat_file::String` : path of the `.dat` file.
"""
function read_dat_file(dat_file::String)
    isfile(dat_file) || throw(ArgumentError(".dat file not found."))

    # Extraction des données individuelles depuis le fichier .dat
    file = readdlm(dat_file, '\n')
    nb_vertices = length(file)-1
    gsm = Dict{Int64,String}() # groupe sanguin du malade
    gsd = Dict{Int64,String}() # groupe sanguin du donneur
    pra = Dict{Int64,Float64}() # PRA du malade
    for line in file[2:end]
        splitted_line = split(line, ',')
        pair = parse(Int, splitted_line[1])
        gsm[pair] = String(splitted_line[2])
        gsd[pair] = String(splitted_line[3])
        pra[pair] = parse(Float64, splitted_line[5])
    end

    # Construction du graphe de compatibilité a priori ̃G ; on fixe le poids des arcs à 1 par défaut (à modifier dans une autre fonction si souhaité) 
    Gtilde = SimpleDiGraph(nb_vertices, 0)
    edge_weight = zeros(nb_vertices,nb_vertices)
    for u in 1:nb_vertices
        for v in u+1:nb_vertices
            if gs_compatible(gsd[u], gsm[v])
                add_edge!(Gtilde, u, v)
                edge_weight[u,v] = 1
            end
            if gs_compatible(gsd[v], gsm[u])
                add_edge!(Gtilde, v, u)
                edge_weight[v,u] = 1
            end
        end
    end

    return Gtilde, edge_weight, gsm, gsd, pra
end

read_dat_file

In [13]:
# Exemple d'utilisation de la fonction supposant que les fichiers sont dans le dossier data
Gtilde, edge_weight, gsm, gsd, pra = read_dat_file("data_KEP/KEP_191.dat");
println("Number of vertices: ", nv(Gtilde));
println("Number of arcs: ", ne(Gtilde));
println("List of arcs with weights:")
for e in edges(Gtilde)
    print("($(e.src),$(e.dst)):$(edge_weight[e.src,e.dst]), ");
end

Number of vertices: 512
Number of arcs: 112203
List of arcs with weights:
(1,3):1.0, (1,4):1.0, (1,8):1.0, (1,12):1.0, (1,14):1.0, (1,15):1.0, (1,17):1.0, (1,25):1.0, (1,31):1.0, (1,32):1.0, (1,37):1.0, (1,41):1.0, (1,44):1.0, (1,48):1.0, (1,49):1.0, (1,58):1.0, (1,59):1.0, (1,64):1.0, (1,65):1.0, (1,66):1.0, (1,69):1.0, (1,72):1.0, (1,74):1.0, (1,77):1.0, (1,83):1.0, (1,89):1.0, (1,90):1.0, (1,92):1.0, (1,93):1.0, (1,96):1.0, (1,97):1.0, (1,100):1.0, (1,103):1.0, (1,107):1.0, (1,110):1.0, (1,115):1.0, (1,119):1.0, (1,124):1.0, (1,125):1.0, (1,132):1.0, (1,139):1.0, (1,150):1.0, (1,154):1.0, (1,156):1.0, (1,159):1.0, (1,161):1.0, (1,165):1.0, (1,168):1.0, (1,175):1.0, (1,176):1.0, (1,177):1.0, (1,182):1.0, (1,183):1.0, (1,186):1.0, (1,187):1.0, (1,189):1.0, (1,192):1.0, (1,193):1.0, (1,194):1.0, (1,196):1.0, (1,197):1.0, (1,203):1.0, (1,206):1.0, (1,208):1.0, (1,212):1.0, (1,215):1.0, (1,221):1.0, (1,222):1.0, (1,227):1.0, (1,230):1.0, (1,231):1.0, (1,233):1.0, (1,240):1.0, (1,245):1.0

Excessive output truncated after 524314 bytes.

(164,49):1.0, (164,58):1.0, (164,59):1.0, (164,64):1.0, (164,65):1.0, (164,66):1.0, (164,69):1.0, (164,72):1.0, (164,74):1.0, (164,77):1.0, (164,83):1.0, (164,89):1.0, (164,90):1.0, (164,92):1.0, (164,93):1.0, (164,96):1.0, (164,97):1.0, (164,100):1.0, (164,103):1.0, (164,107):1.0, (164,110):1.0, (164,115):1.0, (164,119):1.0, (164,124):1.0, (164,125):1.0, (164,132):1.0, (164,139):1.0, (164,150):1.0, (164,154):1.0, (164,156):1.0, (164,159):1.0, (164,161):1.0, (164,165):1.0, (164,168):1.0, (164,175):1.0, (164,176):1.0, (164,177):1.0, (164,182):1.0, (164,183):1.0, (164,186):1.0, (164,187):1.0, (164,189):1.0, (164,192):1.0, (164,193):1.0, (164,194):1.0, (164,196):1.0, (164,197):1.0, (164,203):1.0, (164,206):1.0, (164,208):1.0, (164,212):1.0, (164,215):1.0, (164,221):1.0, (164,222):1.0, (164,227):1.0, (164,230):1.0, (164,231):1.0, (164,233):1.0, (164,240):1.0, (164,245):1.0, (164,248):1.0, (164,249):1.0, (164,253):1.0, (164,254):1.0, (164,257):1.0, (164,258):1.0, (164,259):1.0, (164,260):1.

In [7]:
DISTRIBUTIONS = ["Constant","Binomial","BinomialUNOS","BinomialAPD","NoFailure"]

"""
    get_failure_rates

Generate failure rates on each edge, and add its value as a property to the edge of the kep_graph. 

# Parameters
* `kep_graph::SimpleDiGraph` : graph describing the pairs and compatibilities
* `pra::Dict{Int64,Float64}` : PRA de chaque malade
* `distribution::String` : type of distirbution of uncertainties; to be chosen in the DISTRIBUTIONS vector
"""
function get_failure_rates(kep_graph::SimpleDiGraph, pra::Dict{Int64,Float64}, distribution::String)

    failure_rate = Dict{Tuple{Int64,Int64},Float64}() # probabilité d'échec d'un transfert d'organe du donneur d'une paire vers le patient d'une autre

    for edge in edges(kep_graph)
        # Failure rates depend on the chosen distribution of uncertainties
        if distribution == "Constant"
            # constant failure rates equal to 70%
            failure_rate[edge.src,edge.dst] = 0.7
        elseif distribution == "Binomial"
            if rand() < 0.25
                # random failure rates equal to 10% on average for 25% edges
                failure_rate[edge.src,edge.dst] = rand() * 0.2
            else
                # random failure rates equal to 90% on average for 75% edges
                failure_rate[edge.src,edge.dst] = 0.8 + rand() * 0.2
            end
        elseif distribution == "BinomialUNOS"
            # %pra denotes the panel reactive antibody level
            # %pra of the patient < 0.8 : UNOS low sensitized patients
            if pra[edge.dst] < 0.8
                # failure rate equal to 10% if the patient is low sensitized
                failure_rate[edge.src,edge.dst] = 0.1
            else
                # failure rate equal to 90% otherwise 
                failure_rate[edge.src,edge.dst] = 0.9
            end
        elseif distribution  == "BinomialAPD"
            # %pra denotes the panel reactive antibody level
            # %pra of the patient < 0.75 : APD low sensitized patients
            if pra[edge.dst] < 0.75
                # failure rate equal to 28% if the patient is low sensitized
                failure_rate[edge.src,edge.dst] = 0.28
            else
                # failure rate equal to 58% otherwise 
                failure_rate[edge.src,edge.dst] = 0.58
            end
        elseif distribution == "NoFailure"
            # failure rates equal to 0
            failure_rate[edge.src,edge.dst] = 0.
        end
    end

    return failure_rate
end


get_failure_rates

In [14]:
failure_rates = get_failure_rates(Gtilde, pra, "Binomial")

Dict{Tuple{Int64, Int64}, Float64} with 112203 entries:
  (344, 278) => 0.841089
  (506, 449) => 0.0380436
  (308, 347) => 0.874488
  (301, 90)  => 0.940579
  (141, 20)  => 0.0479984
  (379, 345) => 0.0441052
  (481, 374) => 0.00912038
  (12, 418)  => 0.916105
  (283, 92)  => 0.191814
  (458, 90)  => 0.92122
  (161, 462) => 0.90972
  (48, 61)   => 0.943434
  (363, 209) => 0.925299
  (60, 171)  => 0.919256
  (452, 442) => 0.922628
  (228, 491) => 0.081269
  (260, 244) => 0.869373
  (280, 124) => 0.989313
  (302, 101) => 0.894043
  (401, 374) => 0.853347
  (426, 368) => 0.98724
  (25, 73)   => 0.871904
  (91, 30)   => 0.0202424
  (100, 10)  => 0.812474
  (306, 456) => 0.857406
  ⋮          => ⋮

In [124]:
function exchange(transferts)
    failure_rates = get_failure_rates(Gtilde, pra, "Binomial")
    nb_transferts = 0
    cycles = Vector{Vector{Int}}()
    for c in transferts
        ajout = true
        if rand(Bernoulli(1-failure_rates[(c[end],c[1])]),1) == [0]
            ajout = false
            continue
        end
        for i in 2:length(c)
            if rand(Bernoulli(1-failure_rates[(c[i-1],c[i])]),1) == [0]
                ajout = false
                break
            end
        end
        if ajout
            push!(cycles,c)
            nb_transferts += length(c)
        end
    end
    return cycles
end
println(cycles)
println(nb_transferts)

[[216, 240], [150, 415], [277, 491], [96, 380], [82, 99], [427, 428], [25, 451], [158, 511], [120, 327], [410, 443], [141, 228], [129, 486], [61, 356], [57, 107], [3, 89]]
30


In [126]:
transferts

17-element Vector{Vector{Int64}}:
 [42, 48]
 [10, 17]
 [16, 19]
 [21, 27]
 [5, 43]
 [18, 46]
 [1, 13]
 [7, 44]
 [38, 50]
 [32, 39]
 [11, 41]
 [8, 15]
 [14, 34]
 [25, 40, 29, 26]
 [4, 31]
 [22, 24]
 [28, 35, 30]

In [125]:
exchange(transferts)

LoadError: KeyError: key (17, 10) not found

In [45]:
vlist = shuffle(vertices(G))[1:(32)]

32-element Vector{Int64}:
 18
 12
 64
 50
 22
 44
 53
 60
 48
 21
 29
 26
  4
  ⋮
  3
 55
 33
 14
 49
 20
  9
  1
  8
 27
 42
 16

In [46]:
G_tmp, vmap = induced_subgraph(G, vlist)

(SimpleDiGraph{Int64}(285, [[3, 8, 24, 32], [3, 8, 15, 24, 32], [2, 18, 25, 26, 31], [1, 3, 8, 24, 32], [3, 4, 8, 9, 10, 11, 13, 16, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32], [1, 2, 3, 4, 5, 7, 8, 10, 11, 13  …  19, 21, 22, 23, 24, 25, 27, 28, 31, 32], [1, 3, 8, 24, 32], [2, 18, 25, 26], [2, 20, 25, 26, 31], [3, 8, 24, 32]  …  [3, 8, 15, 24, 32], [2, 3, 4, 5, 7, 8, 9, 10, 11, 13  …  22, 23, 25, 26, 27, 28, 29, 30, 31, 32], [3, 8, 24, 32], [3, 8, 24, 32], [2, 20, 25], [1, 8, 24, 32], [2, 18, 20, 25, 26, 31], [1, 3, 8, 24, 32], [3, 15, 32], Int64[]], [[4, 6, 7, 11, 12, 18, 21, 28, 30], [3, 6, 8, 9, 14, 15, 16, 18, 19, 20, 24, 27, 29], [1, 2, 4, 5, 6, 7, 10, 11, 12, 13  …  18, 20, 21, 22, 23, 24, 25, 26, 30, 31], [5, 6, 15, 16, 18, 20, 24], [6, 14, 15, 16, 18, 20, 24], Int64[], [6, 14, 15, 16, 20, 24], [1, 2, 4, 5, 6, 7, 10, 11, 12, 13  …  18, 20, 21, 22, 23, 24, 25, 26, 28, 30], [5, 14, 15, 16, 18, 20, 24], [5, 6, 14, 15, 16, 18, 20, 24]  …  [5, 6, 14, 15, 16, 18, 20, 24], [1, 2, 

In [47]:
G_tmp

{32, 285} directed simple Int64 graph

In [50]:
setdiff(vertices(G), vlist)

32-element Vector{Int64}:
  2
  5
  6
  7
 10
 11
 13
 15
 17
 19
 23
 24
 25
  ⋮
 40
 43
 45
 47
 51
 52
 54
 56
 57
 58
 59
 62

In [118]:
v_list = sort(shuffle(vertices(G))[1:(32)])
v_valid = setdiff(vertices(G),v_list)
println("Liste d'attente initiale : $v_list")
println("-----------------------------")
for i in 1:6
    if length(v_list) < 2
        break
    end
    v_filter = sort(shuffle(v_list)[1:convert(Int, floor(0.3*length(v_list)))])
    for v in v_filter
        v_list = filter!(e->e≠v,v_list)
        v_valid = filter!(e->e≠v,v_valid)
    end
    v_add = sort(shuffle(setdiff(v_valid, v_list))[1:convert(Int, floor(0.2*length(v_valid)))])
    for v in v_add
        push!(v_list, v)
        v_valid = filter!(e->e≠v,v_valid)
    end
    v_list = sort(v_list)
    println("Nouveaux patients : $v_add")
    println("Patients qui quittent la file d'attente : $v_filter")
    println("Liste d'attente : $v_list")
    println("-----------------------------")
end

Liste d'attente initiale : [2, 7, 8, 10, 12, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 28, 32, 36, 38, 39, 40, 44, 47, 52, 53, 55, 56, 57, 58, 59, 63]
-----------------------------
Nouveaux patients : [6, 33, 35, 49, 61, 62]
Patients qui quittent la file d'attente : [8, 10, 28, 38, 39, 53, 55, 56, 59]
Liste d'attente : [2, 6, 7, 12, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 32, 33, 35, 36, 40, 44, 47, 49, 52, 57, 58, 61, 62, 63]
-----------------------------
Nouveaux patients : [4, 17, 30, 42, 43]
Patients qui quittent la file d'attente : [7, 15, 20, 22, 33, 44, 52, 61]
Liste d'attente : [2, 4, 6, 12, 16, 17, 18, 19, 21, 23, 24, 25, 26, 30, 32, 35, 36, 40, 42, 43, 47, 49, 57, 58, 62, 63]
-----------------------------
Nouveaux patients : [1, 3, 34, 60]
Patients qui quittent la file d'attente : [4, 12, 16, 26, 32, 57, 62]
Liste d'attente : [1, 2, 3, 6, 17, 18, 19, 21, 23, 24, 25, 30, 34, 35, 36, 40, 42, 43, 47, 49, 58, 60, 63]
-----------------------------
Nouveaux patients : [31, 48

In [127]:
G = Gtilde

v_list = sort(shuffle(vertices(G))[1:convert(Int, floor(nv(G)/2))])
v_valid = setdiff(vertices(G),v_list)
println("Liste d'attente initiale : $v_list")
println("-----------------------------")
for i in 1:6
    if length(v_list) < 2
        break
    end
    println("1")
    graph, vmap = induced_subgraph(G,vlist)
    println("2")
    test = KEP_test(graph ; SP_method="Bellmann")
    println("3")
    transferts = solve_KEP(test)[1]
    println("4")
    #transferts ne possède pas les bons indices, i.e. ce sont les indices du sous graphe et pas ceux du graphe initial
    echanges_realises = exchange(transferts)
    println("5")
    v_filter = Vector{Int}()
    for c in echanges_realises
        for v in c
            push!(v_filter, vmap[v])
        end
    end
    v_filter = sort(v_filter)
    for v in v_filter
        v_list = filter!(e->e≠v,v_list)
        v_valid = filter!(e->e≠v,v_valid)
    end
    
    v_add = sort(shuffle(setdiff(v_valid, v_list))[1:convert(Int, floor(0.2*length(v_valid)))])
    for v in v_add
        push!(v_list, v)
        v_valid = filter!(e->e≠v,v_valid)
    end
    v_list = sort(v_list)
    println("Nouveaux patients : $v_add")
    println("Patients qui quittent la file d'attente : $v_filter")
    println("Liste d'attente : $v_list")
    println("-----------------------------")
end

Liste d'attente initiale : [1, 5, 6, 7, 8, 9, 10, 13, 15, 16, 21, 23, 25, 26, 28, 30, 34, 35, 36, 37, 38, 40, 52, 53, 55, 56, 58, 60, 62, 66, 67, 68, 72, 74, 75, 77, 78, 81, 82, 85, 90, 91, 92, 93, 94, 99, 102, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 116, 121, 124, 125, 126, 127, 128, 133, 134, 135, 140, 141, 142, 144, 145, 146, 147, 149, 158, 160, 161, 163, 164, 165, 167, 168, 172, 174, 176, 178, 182, 185, 188, 189, 196, 201, 205, 206, 207, 209, 210, 211, 213, 216, 218, 219, 221, 222, 223, 224, 227, 228, 230, 232, 233, 236, 237, 238, 239, 240, 241, 242, 244, 248, 249, 251, 252, 253, 254, 255, 257, 258, 259, 260, 266, 268, 273, 276, 278, 280, 282, 283, 285, 287, 288, 289, 290, 293, 295, 300, 301, 303, 304, 306, 308, 309, 312, 313, 314, 315, 318, 319, 320, 321, 322, 327, 329, 330, 331, 334, 335, 342, 343, 344, 345, 346, 347, 351, 352, 353, 354, 357, 358, 359, 362, 363, 365, 369, 371, 374, 376, 381, 384, 385, 387, 390, 391, 395, 396, 397, 398, 400, 401, 402, 404, 405, 406, 407,

LoadError: KeyError: key (41, 23) not found